In [ ]:
import sys
import numpy as np

In [ ]:
from tqdm import tqdm
import torch.nn.functional as F
import torch
from transformers import AutoTokenizer, AutoModel
sys.path.insert(0, '../src')
from schema import Document, Chunk
from cadec_loader import load_cadec
from chunker import chunk_document
from embedder import encode


In [3]:
# load doc and create chunks
docs = load_cadec("../data/cadec/")

all_chunks=[]
for doc in docs:
    chunk = chunk_document(doc, chunk_size =400, overlap =1)
    all_chunks.extend(chunk)

print("Total chunk: ", len(all_chunks))

Loaded 1220 documents from ../data/cadec/
Total chunk:  2549


In [4]:
MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

print("Loading tokenizer...")
tokenizer= AutoTokenizer.from_pretrained(MODEL_NAME)
print("Loading model...")
model = AutoModel.from_pretrained(MODEL_NAME)

model.eval() # Disable dropout. We want determinsitic output at inference.

print("Model loaded.")
print("Modle hidden emdding size: ", model.config.hidden_size)
print("Max position embeddings: ", model.config.max_position_embeddings)

Loading tokenizer...
Loading model...
Model loaded.
Modle hidden emdding size:  768
Max position embeddings:  512


In [5]:
print(all_chunks[1].text)

Due to my arthritis getting progressively worse, to the point where I am in tears with the agony, gp's started me on 75 twice a day and I have to take it.
every day for the next month to see how I get on, here goes. So far its been very good, pains almost gone, but I feel a bit weird, didn't have that when on 50.


In [20]:
def encode(texts: list, batch_size: int)-> np.ndarray:
    all_embeddings= []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        # step 1 = tokenize
        encoded = tokenizer(
            batch,
            truncation= True,
            padding = True,
            max_length= 512,
            return_tensors = "pt"
        )

        # step 2 = Forward Pass
        with torch.no_grad():
            outputs = model(**encoded)

        # step 3 = mean pooling
        token_embeddings= outputs.last_hidden_state   # [batch_size, seq_len, 768]
        attention_mask = encoded["attention_mask"]    # [batch_size, seq_len]

        attention_mask_expanded = attention_mask.unsqueeze(-1).float()  # [batch_size, seq_len, 1]
        sum_embeddings = (token_embeddings*attention_mask_expanded).sum(dim=1)  # [batch_size, 768]
        sum_mask = attention_mask_expanded.sum(dim =1).clamp(min = 1e-9)        # [batch_size, 1]

        pooled = sum_embeddings/sum_mask    # [batch_size, 768]


        # step 4 = L2 Normalize
        normed = F.normalize(pooled, p=2, dim=1)   # [batch, 768]


        all_embeddings.append(normed.numpy())


    return np.vstack(all_embeddings)

In [21]:
texts = [chunk.text for chunk in all_chunks]

print(f"Embedding {len(texts)} chunks...")
embeddings = encode(texts, batch_size = 32)

print(f"Embedding matrix: {embeddings.shape}")


Embedding 2549 chunks...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 80/80 [00:56<00:00,  1.41it/s]

Embedding matrix: (2549, 768)


In [23]:
# Verify L2 norms are all 1.0
norms = np.linalg.norm(embeddings, axis=1)
print(f"\nVector norms — min: {norms.min():.4f}, max: {norms.max():.4f}")
print("All unit vectors:", np.allclose(norms, 1.0, atol=1e-5))



Vector norms — min: 1.0000, max: 1.0000
All unit vectors: True


In [24]:
np.save("../index/embeddings.npy", embeddings)
print(f"Saved embeddings {embeddings.shape} -> '../index/embeddings.npy'")

Saved embeddings (2549, 768) -> '../index/embeddings.npy'


In [26]:
loaded = np.load("../index/embeddings.npy")
print(f"Loaded embeddings: {np.allclose(embeddings, loaded)}")

Loaded embeddings: True


In [34]:
# Pick two chunks about the same ADR — should have high cosine similarity
# Pick two chunks about different things — should have low similarity

def cosine_sim(a, b):
    # Since vectors are already L2 normalised, dot product = cosine similarity
    return float(np.dot(a, b))

# Find a chunk clearly about muscle pain / statins
statin_chunk = next(c for c in all_chunks 
                    if "muscle" in c.text.lower() 
                    and c.drug_name == "atorvastatin")

# Find a chunk clearly about something unrelated — stomach/GI issues
stomach_chunk = next(c for c in all_chunks 
                     if "stomach" in c.text.lower() 
                     and c.drug_name == "atorvastatin")

# Find a chunk about cholesterol reduction (positive outcome, different topic)
cholesterol_chunk = next(c for c in all_chunks 
                         if "cholesterol" in c.text.lower() 
                         and "pain" not in c.text.lower()
                         and c.drug_name == "atorvastatin")

idx_a = all_chunks.index(statin_chunk)
idx_b = all_chunks.index(stomach_chunk)
idx_c = all_chunks.index(cholesterol_chunk)

sim_same= cosine_sim(embeddings[idx_a], embeddings[idx_b])
sim_cross = cosine_sim(embeddings[idx_a], embeddings[idx_c])

print(f"Chunk A: {statin_chunk.text[:300]}...")
print(f"Chunk B: {stomach_chunk.text[:300]}...")
print(f"Chunk C: {cholesterol_chunk.text[:300]}...")
print()
print(f"Similarity A↔B (same topic) : {sim_same:.4f}")
print(f"Similarity A↔C (diff topic) : {sim_cross:.4f}")
print()
print("A↔B should be noticeably higher than A↔C")


Chunk A: Muscle pain, leg cramps, fatigue, irritable, pain in feet, neuropathy.
I have stopped this medication a couple of times and my doctor keeps insisting I take it, rather then alternative medications.
I'm getting off of this stuff after reading so many people are having similiar side-effects.
Doctors n...
Chunk B: ALL AND SOME NOT NOTED - KIDNEY-METAL MOODS-ANZITY-SUGAR LEVEL DROPS - STOMACH ULCER-JOINT PAIN- MEMORY TROUBLE- PUT ON MORE MEDS- LIPITOR ADMITED TO GIVING KIDNEY TROUBLES TO ME -I TOOK OTHER S WITH SAME TROUBLES AND MORE....
Chunk C: Diet and exercise does not work for my genetic high cholesterol, but I am with the majority here and it's not worth it if ending up crippled or senile. I will find a natural alternative. I wish you all the best. I wish you all the best....

Similarity A↔B (same topic) : 0.9837
Similarity A↔C (diff topic) : 0.9886

A↔B should be noticeably higher than A↔C


In [ ]:
# ADR chunk
adr_chunk = next(c for c in all_chunks 
                 if "muscle" in c.text.lower()
                 and c.drug_name == "atorvastatin")

# Most different thing possible — a very short neutral chunk
neutral_chunk = next(c for c in all_chunks
                     if "effective" in c.text.lower()
                     and len(c.text) < 60)

idx_a = all_chunks.index(adr_chunk)
idx_b = all_chunks.index(neutral_chunk)

print(f"ADR chunk     : {adr_chunk.text[:80]}")
print(f"Neutral chunk : {neutral_chunk.text[:80]}")
print(f"Similarity    : {cosine_sim(embeddings[idx_a], embeddings[idx_b]):.4f}")

# Now compare a query vector against both
query = "What are the muscle side effects of atorvastatin?"
query_embedding = encode([query])  # shape [1, 768]

sim_query_adr     = cosine_sim(query_embedding[0], embeddings[idx_a])
sim_query_neutral = cosine_sim(query_embedding[0], embeddings[idx_b])

print(f"\nQuery: '{query}'")
print(f"Similarity to ADR chunk     : {sim_query_adr:.4f}")
print(f"Similarity to neutral chunk : {sim_query_neutral:.4f}")
print(f"\nADR chunk should score higher against the query.")

In [36]:
# ADR chunk
adr_chunk = next(c for c in all_chunks 
                 if "muscle" in c.text.lower()
                 and c.drug_name == "atorvastatin")

# Most different thing possible — a very short neutral chunk
neutral_chunk = next(c for c in all_chunks
                     if "effective" in c.text.lower()
                     and len(c.text) < 60)

idx_a = all_chunks.index(adr_chunk)
idx_b = all_chunks.index(neutral_chunk)

print(f"ADR chunk     : {adr_chunk.text[:80]}")
print(f"Neutral chunk : {neutral_chunk.text[:80]}")
print(f"Similarity    : {cosine_sim(embeddings[idx_a], embeddings[idx_b]):.4f}")

# Now compare a query vector against both
query = "What are the muscle side effects of atorvastatin?"
query_embedding = encode([query], batch_size=32)  # shape [1, 768]

sim_query_adr     = cosine_sim(query_embedding[0], embeddings[idx_a])
sim_query_neutral = cosine_sim(query_embedding[0], embeddings[idx_b])

print(f"\nQuery: '{query}'")
print(f"Similarity to ADR chunk     : {sim_query_adr:.4f}")
print(f"Similarity to neutral chunk : {sim_query_neutral:.4f}")
print(f"\nADR chunk should score higher against the query.")

ADR chunk     : Muscle pain, leg cramps, fatigue, irritable, pain in feet, neuropathy.
I have st
Neutral chunk : Very effective.
Lowered cholesterol from 230 to 160.
Similarity    : 0.9659


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 35.17it/s]


Query: 'What are the muscle side effects of atorvastatin?'
Similarity to ADR chunk     : 0.9696
Similarity to neutral chunk : 0.9597

ADR chunk should score higher against the query.
